# [11.1] PCA, SVD, and Geometry Controls - Solutions

Runs the local reference implementation, all visible tests, and the committed CUDA verification-report assertions.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter11_representation_geometry"
section = "part1_pca_svd_geometry_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

from chapter11_representation_geometry.exercises.part1_pca_svd_geometry_controls import solutions
import part1_pca_svd_geometry_controls.tests as tests


## Visible tests

In [ ]:
tests.test_pca_svd_projection_centers_and_reports_variance(
    solutions.pca_svd_projection,
)
tests.test_geometry_label_prediction_report_uses_heldout_centroids(
    solutions.geometry_label_prediction_report,
)
tests.test_white_noise_control_report_requires_margin(
    solutions.white_noise_control_report,
)
tests.test_geometry_stability_report_averages_pairwise_jaccard(
    solutions.geometry_stability_report,
)
tests.test_direction_causal_effect_report_beats_random_control(
    solutions.direction_causal_effect_report,
)
tests.test_template_center_activations_removes_each_template_mean(
    solutions.template_center_activations,
)
tests.test_notebook_contract(solutions.run_smoke_test)

## Smoke-test contract

In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
assert smoke["pca"]["projected_shape"] == [4, 1]
assert smoke["pca"]["explained_variance_ratio"] == [1.0]
assert smoke["prediction"]["heldout_accuracy"] == 1.0
assert smoke["noise_control"]["survives_white_noise_control"] is True
assert smoke["stability"]["stable_across_seeds"] is True
assert smoke["causal_direction"]["has_causal_effect"] is True
assert smoke["template_centering"]["max_template_mean_abs"] == 0.0
smoke

## Committed CUDA verification report

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"] is True
assert report["tests_passed"] is True
assert report["gt_tier"] == "GT-0"
assert gpu["cuda_available"] is True
assert gpu["predicts_heldout_labels"] is True
assert gpu["template_centering_max_mean_abs"] == 0.0
assert gpu["pythia_calendar_preflight_passed"] is True
assert gpu["pythia_calendar_task_count"] == 2
assert gpu["pythia_weekday_generation_used"] is False
assert gpu["pythia_weekday_raw_heldout_accuracy"] <= 0.2
assert gpu["pythia_weekday_centered_heldout_accuracy"] == 1.0
assert gpu["pythia_weekday_permuted_label_accuracy"] == 0.0
assert gpu["pythia_weekday_noise_accuracy"] <= 0.2
assert gpu["pythia_weekday_matched_pair_accuracy"] == 1.0
assert gpu["pythia_month_raw_heldout_accuracy"] <= 0.75
assert gpu["pythia_month_centered_heldout_accuracy"] == 1.0
assert gpu["pythia_month_permuted_label_accuracy"] == 0.0
assert gpu["pythia_month_noise_accuracy"] <= 0.1
assert gpu["pythia_month_matched_pair_accuracy"] == 1.0
assert gpu["within_vram_budget"] is True
assert report["peak_vram_gb"] <= report["baselines"]["expected_metrics"]["max_allowed_gpu_gb"]
{
    "gpu_name": report["gpu_name"],
    "peak_vram_gb": report["peak_vram_gb"],
    "weekday_centered_accuracy": gpu["pythia_weekday_centered_heldout_accuracy"],
    "month_centered_accuracy": gpu["pythia_month_centered_heldout_accuracy"],
}